# Linear Regression

This notebook accompanies the **ML Viz** lesson on linear regression.
We'll implement OLS, Ridge, and Lasso from scratch.

**Companion lesson:** https://ml-viz.vercel.app/courses/linear-regression/01-linear-regression

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = '#94a3b8'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#2e3347'

## OLS: Closed-Form Solution

$$w^* = (X^T X)^{-1} X^T y$$

In [ ]:
np.random.seed(42)
n = 100
X = 2 * np.random.randn(n, 1) + 1
y = 3.5 * X.squeeze() + 1.2 + 0.5 * np.random.randn(n)

# OLS
X_b = np.c_[np.ones(n), X]  # add bias column
w_ols = np.linalg.inv(X_b.T @ X_b) @ X_b.T @ y
print(f'OLS: w = {w_ols[1]:.3f}, b = {w_ols[0]:.3f}')

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(X, y, c='#818cf8', s=15, alpha=0.6, label='Data')
x_line = np.linspace(X.min(), X.max(), 100)
ax.plot(x_line, w_ols[0] + w_ols[1] * x_line, color='#14b8a6', linewidth=2, label='OLS fit')
ax.legend()
ax.set_title('Ordinary Least Squares', color='white')
plt.tight_layout()
plt.show()

## Ridge vs Lasso

Ridge (L2): $\mathcal{L} = \|y - Xw\|^2 + \lambda\|w\|^2$
Lasso (L1): $\mathcal{L} = \|y - Xw\|^2 + \lambda\|w\|_1$

In [ ]:
def ridge(X, y, lam):
    n = X.shape[0]
    return np.linalg.inv(X.T @ X + lam * np.eye(X.shape[1])) @ X.T @ y

def lasso.coordinate_descent(X, y, lam, n_iter=1000):
    w = np.zeros(X.shape[1])
    for _ in range(n_iter):
        for j in range(X.shape[1]):
            r = y - X @ w + X[:, j] * w[j]
            z_j = X[:, j] @ X[:, j]
            w[j] = np.sign(X[:, j] @ r / len(y)) * max(abs(X[:, j] @ r / len(y)) - lam, 0) / z_j * len(y)
    return w

lambdas = [0.01, 0.1, 1.0, 5.0]
fig, axes = plt.subplots(1, len(lambdas), figsize=(4 * len(lambdas), 4), sharey=True)
fig.suptitle('Ridge Regression: Effect of λ', color='white', fontsize=13, y=1.02)

for ax, lam in zip(axes, lambdas):
    w_r = ridge(X_b, y, lam)
    ax.scatter(X, y, c='#818cf8', s=10, alpha=0.4)
    ax.plot(x_line, w_r[0] + w_r[1] * x_line, color='#14b8a6', linewidth=2)
    ax.set_title(f'λ = {lam}', color='white', fontsize=11)
plt.tight_layout()
plt.show()

## Worked example: OLS by hand

From the lesson: $X = [[1,1],[1,2],[1,3]]$, $y = [2,3,5]$. Solve $w^* = (X^\top X)^{-1} X^\top y$.

In [ ]:
X = np.array([[1, 1], [1, 2], [1, 3]])
y = np.array([2, 3, 5])
w = np.linalg.inv(X.T @ X) @ X.T @ y
print('w* =', w.round(3), '  (intercept, slope)')

## Regularization paths: Ridge vs Lasso

As the penalty $\alpha$ grows, Ridge shrinks weights smoothly toward zero; Lasso drives some to **exactly** zero (feature selection).

In [ ]:
from sklearn.linear_model import Ridge, Lasso
from sklearn.datasets import make_regression

Xr, yr = make_regression(n_samples=100, n_features=8, n_informative=3,
                         noise=10, random_state=0)
alphas = np.logspace(-2, 2, 30)
ridge = np.array([Ridge(a).fit(Xr, yr).coef_ for a in alphas])
lasso = np.array([Lasso(a).fit(Xr, yr).coef_ for a in alphas])

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(alphas, ridge); ax[0].set_xscale('log'); ax[0].set_title('Ridge (L2)')
ax[1].plot(alphas, lasso); ax[1].set_xscale('log'); ax[1].set_title('Lasso (L1)')
for a in ax: a.set_xlabel('alpha'); a.set_ylabel('coefficient')
plt.tight_layout(); plt.show()

## Key takeaways

- Linear regression fits $y = Xw$ by minimizing squared error.
- **OLS** has a closed form: $w^* = (X^\top X)^{-1} X^\top y$.
- **Ridge (L2)** shrinks weights smoothly; **Lasso (L1)** zeros some out (sparse models).
- Regularization trades a little bias for much lower variance — key against overfitting.